# Jobs A 2025 Dataset - Cleaning Notebook

## Step 0: Load & Initial Reconnaissance
This notebook handles initial data loading and cleaning decisions.


In [12]:
import pandas as pd

# Load raw dataset (adjust filename if needed)

df = pd.read_csv("../data/raw/dfall_clean.csv", parse_dates = ['created_dt', 'created_date'])

# Quick look
print(df.shape)
print(df.dtypes)
df.head()

(4000, 20)
id                                   int64
title                               object
company_name                        object
category_label                      object
category_tag                        object
country                             object
location_display                    object
location_area                       object
latitude                           float64
longitude                          float64
contract_type                       object
contract_time                       object
salary_min                         float64
salary_max                         float64
salary_mid                         float64
salary_is_predicted                  int64
created_dt             datetime64[ns, UTC]
created_date                datetime64[ns]
redirect_url                        object
adref                               object
dtype: object


,id,title,company_name,category_label,category_tag,country,location_display,location_area,latitude,longitude,contract_type,contract_time,salary_min,salary_max,salary_mid,salary_is_predicted,created_dt,created_date,redirect_url,adref
0,5410474609,Mechanical Pipefitter,Dodd Group,Trade & Construction Jobs,trade-construction-jobs,United Kingdom,"King's Lynn, Norfolk",UK > Eastern England > Norfolk > King's Lynn,52.751999,0.395357,permanent,full_time,40518.74,40518.74,40518.74,1,2025-09-22 09:28:13+00:00,2025-09-22,https://www.adzuna.co.uk/jobs/details/54104746...,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTQxMDQ3NDYwOSIsI...
1,5381519011,International Trade Policy Editor [Startup Pol...,Applio Ventures,Graduate Jobs,graduate-jobs,United Kingdom,"London, UK",UK > London,NaN,NaN,permanent,full_time,64000.00,64000.00,64000.00,0,2025-09-03 21:20:59+00:00,2025-09-03,https://www.adzuna.co.uk/jobs/details/53815190...,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTM4MTUxOTAxMSIsI...
2,5389371380,Domestic Cleaner,Maid2Clean Suffolk Ltd,Domestic help & Cleaning Jobs,domestic-help-cleaning-jobs,United Kingdom,"Diss, Norfolk",UK > Eastern England > Norfolk > Diss,52.377602,1.106380,permanent,part_time,13.00,13.00,13.00,0,2025-09-08 15:49:25+00:00,2025-09-08,https://www.adzuna.co.uk/jobs/details/53893713...,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiRWtwZjc5cWI4QkdZM...
3,5403097985,Early Years BABY Room Leader,Jesters Childcare Essex,Teaching Jobs,teaching-jobs,United Kingdom,UK,UK,NaN,NaN,permanent,full_time,26707.00,29494.00,28100.50,0,2025-09-17 15:00:12+00:00,2025-09-17,https://www.adzuna.co.uk/jobs/details/54030979...,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTQwMzA5Nzk4NSIsI...
4,5400306117,Senior Back-End Engineer,Eligible Limited,IT Jobs,it-jobs,United Kingdom,UK,UK,NaN,NaN,permanent,full_time,80000.00,NaN,80000.00,0,2025-09-15 17:51:23+00:00,2025-09-15,https://www.adzuna.co.uk/jobs/details/54003061...,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiRWtwZjc5cWI4QkdZM...


## Step 1: Code Assessment
This step reviews entries in the code to highlight suspicious entries

In [57]:
print(f'Number of NaN entries in each column:\n{df.isna().sum()}')

'''Review salary columns'''
df_test_salary = df[(df['salary_min'].isna()==True) & (df['salary_max'].isna()==True) & (df['salary_min'].isna() == True)]
print(f'\nNumber of times that salary min, max, and mid values are all missing: {len(df_test_salary)}')
df_test['salary_is_predicted'].unique()
df_test_predict = df[(df['salary_is_predicted'] == 0)]
df_test_predict

'''Check duplicate entries'''
df_dup_id = df[(df['id'].duplicated() == True)]
print(f'\nNumber of ID entries duplicated: {df['id'].duplicated().sum()}')

Number of NaN entries in each column:
id                        0
title                     0
company_name            214
category_label            0
category_tag              0
country                   0
location_display          0
location_area             0
latitude                743
longitude               743
contract_type          3154
contract_time          3015
salary_min             2456
salary_max             2465
salary_mid             2456
salary_is_predicted       0
created_dt                0
created_date              0
redirect_url              0
adref                     0
dtype: int64

Number of times that salary min, max, and mid values are all missing: 2456

Number of ID entries duplicated: 14


Missing data points found in relevant columns-`salary_min`, `salary_max`, `salary_mid`-as well as in columns describing contract types and locations. Comparing missing entries to the `salary_predicted` columns, it is not dependant on whether the other salary columns are filled or not.

An important note is that the number of `NaN` entries in `salary_min` and `salary_mid` are equal, while `salary_max` has 9 more missing entries. This tells us that `salary_mid` is only reliant on `salary_mid` to calculate a score. We may need to ignore the 9 extra `NaN` entries in `salary_max`

There are 14 `id` entries that are duplicate entries so these must be removed from the dataframe

## Step 2: Cleaning Decisions
This step resolves problems discovered in Step 1